# Ordered Logistic Regression Results: A FAIR^2 Dataset Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset—“Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya”—using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

## Dataset Source
The dataset is described using the [Croissant schema](https://mlcommons.org/croissant/) and is accessible via:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```

We'll follow the Croissant structure to explore all record sets, fields, and values by their `@id`.

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and view its description using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Show name and description
print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", metadata.identifier)
print("Version:", metadata.version)
print("License:", metadata.license)

## 2. Data Overview
Examine all available record sets and their fields/columns, referenced strictly by their `@id`.

> **Note:** The `metadata.record_sets` property lists all record sets discovered from the Croissant schema. Each record set (`rec`) has an `@id`, and contains one or more fields defined in `rec.fields`. Each field also has a unique `@id`.

In [ ]:
# List all record sets (@id) and the fields (@id) within each

record_sets = metadata.record_sets
if not record_sets:
    print("No record sets defined in the Croissant schema.")
else:
    for rec in record_sets:
        print(f"RecordSet @id: {rec['@id']}")
        if 'field' in rec and rec['field']:
            field_list = rec['field']
            print("  Fields:")
            for f in field_list:
                if isinstance(f, dict) and '@id' in f:
                    print(f"    Field @id: {f['@id']}")
                else:
                    print(f"    Field reference: {f}")
        else:
            print("  [No fields declared]")

### Retrieve sample data records for a record set
Let's pick the first record set (by `@id`) and list the first three records, referencing the underlying field `@id`s.

In [ ]:
# Choose the first available record set for demonstration

if not record_sets:
    print('No record sets available to preview records.')
else:
    record_set_id = record_sets[0]['@id']
    print(f"Sample records from RecordSet @id = {record_set_id}:")
    # Print 3 records by @id fields
    for i, row in enumerate(dataset.records(record_set=record_set_id)):
        print(f"Record {i+1}: {row}")
        if i >= 2: break

## 3. Data Extraction
Convert all records from each record set into pandas DataFrames for analysis. Reference each record set by its `@id`.

In [ ]:
dataframes = dict()
record_set_ids = [rec['@id'] for rec in record_sets] if record_sets else []

for rid in record_set_ids:
    recs = list(dataset.records(record_set=rid))
    if recs:
        df = pd.DataFrame(recs)
        dataframes[rid] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rid}")
    else:
        print(f"No records found for RecordSet @id: {rid}")

if dataframes:
    first_rset = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for RecordSet @id: {first_rset}")
    print(list(dataframes[first_rset].columns))
    print(dataframes[first_rset].head())
else:
    print('No tabular dataframes extracted.')

## 4. Exploratory Data Analysis (EDA)
We will:
- Filter records based on a numeric field using its `@id`.
- Normalize the filtered values.
- Optionally group by a categorical field (using `@id`).

Refer to [dataset documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json) or the previous code outputs to select appropriate field `@id` values if needed.

> **Note:** Replace `<numeric_field_id>` and `<group_field_id>` below with valid field `@id`s from your loaded DataFrame columns.

In [ ]:
import numpy as np

# EDA will use the first DataFrame if any was loaded
if not dataframes:
    print('No dataframes loaded from record sets. Cannot run EDA.')
else:
    # Select record set id and DataFrame
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Exploring DataFrame for RecordSet @id: {record_set_id}")
    print(df.columns.tolist())
    
    # TRY to auto-select a likely numeric field based on dtype; fallback to the first column
    possible_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    numeric_field_id = possible_numeric[0] if possible_numeric else df.columns[0]
    print(f"Using numeric field for filtering: {numeric_field_id}")
    
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold] if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else df
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by the first non-numeric field if present
    group_fields = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    if group_fields:
        group_field_id = group_fields[0]
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped means by {group_field_id}:")
        print(grouped_df.head())
    else:
        print('No categorical fields available for grouping.')

## 5. Visualization
Now we plot the distribution of the selected numeric field and, if grouped, compare means across categories.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not dataframes:
    print('No data available for visualization.')
else:
    df = dataframes[record_set_id]
    if numeric_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(df[numeric_field_id].dropna(), bins=20)
        plt.title(f"Distribution of {numeric_field_id}")
        plt.xlabel(numeric_field_id)
        plt.ylabel('Frequency')
        plt.show()
    if group_fields:
        # Boxplot/group means
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we've loaded and explored the FAIR^2 dataset using `mlcroissant`, reviewed record sets and field structures by `@id`, extracted representative tables, applied filtering and normalization, and visualized feature distributions.

- **Remember:** For production/advanced analysis, always inspect the Croissant schema and documentation to correctly map data semantics to column `@id`s, especially for filtering/grouping.
- For additional context, ethical considerations, and variable descriptions, consult the full [FAIR^2 dataset documentation](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).

*Generated with [mlcroissant](https://github.com/mlcommons/croissant).*